<a href="https://colab.research.google.com/github/Sharifulislam25/phishing-detector-01/blob/main/phishing_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

print(os.listdir("/content"))

['.config', 'Dataset.csv', 'malicious_phish.csv', 'balanced_urls.csv', 'phishing_site_urls.csv', 'sample_data']


In [2]:
import pandas as pd

In [3]:
df1 = pd.read_csv("/content/phishing_site_urls.csv")

df2 = pd.read_csv("/content/malicious_phish.csv")

df3 = pd.read_csv("/content/balanced_urls.csv")

df4 = pd.read_csv("/content/Dataset.csv")


In [4]:
print(df1.head())

print(df2.head())

print(df3.head())

print(df4.head())

                                                 URL Label
0  nobell.it/70ffb52d079109dca5664cce6f317373782/...   bad
1  www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...   bad
2  serviciosbys.com/paypal.cgi.bin.get-into.herf....   bad
3  mail.printakid.com/www.online.americanexpress....   bad
4  thewhiskeydregs.com/wp-content/themes/widescre...   bad
                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement
                         url   label  result
0     https://www.google.com  benign       0
1    https://www.youtube.com  benign       0
2   https://www.facebook.com  benign       0
3      https://www.baidu.com  benign       0
4  https://www.wikipedia.org  b

In [5]:
!pip install transformers datasets accelerate
!pip install sentence-transformers
!pip install scikit-learn

In [6]:
!pip install transformers datasets accelerate
!pip install sentence-transformers
!pip install scikit-learn

In [7]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)


In [9]:
def clean_columns(df):

    df.columns = [
        str(c).strip().lower()
        for c in df.columns
    ]

    return df

df1 = clean_columns(df1)
df2 = clean_columns(df2)
df3 = clean_columns(df3)
df4 = clean_columns(df4)

In [10]:
print(df1.columns)

print(df2.columns)

print(df3.columns)

print(df4.columns)

Index(['url', 'label'], dtype='object')
Index(['url', 'type'], dtype='object')
Index(['url', 'label', 'result'], dtype='object')
Index(['url', 'url_len', 'dom', 'dom_len', 'is_ip', 'tld', 'tld_len',
       'subdom_cnt', 'letter_cnt', 'digit_cnt', 'special_cnt', 'eq_cnt',
       'qm_cnt', 'amp_cnt', 'dot_cnt', 'dash_cnt', 'under_cnt', 'letter_ratio',
       'digit_ratio', 'spec_ratio', 'is_https', 'slash_cnt', 'entropy',
       'path_len', 'query_len', 'label'],
      dtype='object')


In [11]:
df1["label"] = (
    df1["label"]
    .astype(str)
    .str.lower()
)

df1["label"] = df1["label"].map({
    "good":0,
    "bad":1
})

df1 = df1.dropna()

df1["label"] = df1["label"].astype(int)

df1 = df1[["url","label"]]

In [12]:
mapping = {

    "benign":0,

    "legitimate":0,

    "safe":0,

    "phishing":1,

    "malware":1,

    "defacement":1,

    "spam":1,

    "scam":1
}

df2["type"] = (
    df2["type"]
    .astype(str)
    .str.lower()
)

df2["label"] = df2["type"].map(mapping)

df2 = df2.dropna()

df2["label"] = df2["label"].astype(int)

df2 = df2[["url","label"]]

In [15]:
print(df3.columns)


Index(['url', 'label'], dtype='object')


In [16]:
df3 = df3[["url","label"]]


In [17]:
master_df = pd.concat([

    df1,
    df2,
    df3

], ignore_index=True)

In [18]:
master_df = pd.concat([

    df1,

    df2,

    df3

], ignore_index=True)

In [19]:
master_df = master_df.dropna()

master_df["url"] = (
    master_df["url"]
    .astype(str)
)

master_df = master_df[
    master_df["url"].str.len() > 5
]

master_df = master_df.drop_duplicates(
    subset=["url"]
)

master_df = master_df.sample(
    frac=1,
    random_state=42
)

In [20]:
print(master_df.head())

print(master_df.shape)

print(master_df["label"].value_counts())

                                                       url  label
176816   en.wikipedia.org/wiki/Kansas_City_Fire_Department      0
1431209  https://www.namebase.org/main1/Anthony-_28tony...      0
420301   ratesupermarket.ca/credit_cards/supplier/Laure...      0
939330        http://www.smos.cn/news_show.asp?news_id=104      1
1601099  http://www.ozziedeals.com.au/img/identifient/5...      1
(1127032, 2)
label
0    744522
1    382510
Name: count, dtype: int64


In [21]:
master_df = master_df.sample(

    50000,

    random_state=42
)

In [22]:
print(master_df.shape)

(50000, 2)


In [23]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(

    master_df["url"].tolist(),

    master_df["label"].tolist(),

    test_size=0.2,

    random_state=42
)

In [24]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [25]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

In [26]:
train_encodings = tokenizer(

    train_texts,

    truncation=True,

    padding=True,

    max_length=128
)

test_encodings = tokenizer(

    test_texts,

    truncation=True,

    padding=True,

    max_length=128
)

In [27]:
import torch

class URLDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings

        self.labels = labels

    def __getitem__(self, idx):

        item = {

            key: torch.tensor(val[idx])

            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [28]:
train_dataset = URLDataset(

    train_encodings,

    train_labels
)

test_dataset = URLDataset(

    test_encodings,

    test_labels
)

In [29]:
model = AutoModelForSequenceClassification.from_pretrained(

    model_name,

    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
training_args = TrainingArguments(

    output_dir="./results",

    num_train_epochs=2,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    save_strategy="epoch",

    logging_dir="./logs",

    logging_steps=100
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [32]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset
)


In [ ]:
trainer.train()

Step,Training Loss
100,0.309143
200,0.192040
300,0.163395
400,0.175821
500,0.096800
600,0.107112
700,0.128199
800,0.139372
900,0.097878
1000,0.118154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]